#Load Dataset & Understand it

In [52]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/tweet_product_company.csv")  # change this path

# View first 5 rows
df

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion
...,...,...,...
9088,Ipad everywhere. #SXSW {link},iPad,Positive emotion
9089,"Wave, buzz... RT @mention We interrupt your re...",NaN,No emotion toward brand or product
9090,"Google's Zeiger, a physician never reported po...",NaN,No emotion toward brand or product
9091,Some Verizon iPhone customers complained their...,NaN,No emotion toward brand or product


In [53]:
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


In [3]:
df.columns

Index(['tweet_text', 'emotion_in_tweet_is_directed_at',
       'is_there_an_emotion_directed_at_a_brand_or_product'],
      dtype='object')

In [4]:
df.isnull().sum()

,0
tweet_text,1
emotion_in_tweet_is_directed_at,5802
is_there_an_emotion_directed_at_a_brand_or_product,0


In [5]:
## Drop unnecessary column
df = df.drop("emotion_in_tweet_is_directed_at", axis=1)

For better readability, lets rename column names to 'text' and 'sentiment'.


In [6]:
df.columns = ["text", "sentiment"]

In [7]:
df.head()

,text,sentiment
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Positive emotion


In [8]:
#check for null values again
df.isnull().sum()

,0
text,1
sentiment,0


In [10]:
round(df.isnull().mean()*100,2)

,0
text,0.01
sentiment,0.00


In [11]:
#drop the row
df = df.dropna()

In [12]:
df.isnull().sum()

,0
text,0
sentiment,0


#Text Preprocessing

In [ ]:
#import libraries

In [13]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
#Create Cleaning Function

In [14]:
def clean_text(text):
    text = text.lower()  # lowercase

    # remove URLs, mentions, hashtags
    text = re.sub(r'http\S+|@\w+|#\w+', '', text)

    # remove special characters & numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # remove stopwords
    words = text.split()
    words = [w for w in words if w not in stop_words]

    return " ".join(words)

In [15]:
#Apply Cleaning

In [16]:
df['clean_text'] = df['text'].apply(clean_text)

In [17]:
#Verify Output

In [18]:
df[['text', 'clean_text']].head()

,text,clean_text
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,g iphone hrs tweeting dead need upgrade plugin...
1,@jessedee Know about @fludapp ? Awesome iPad/i...,know awesome ipadiphone app youll likely appre...
2,@swonderlin Can not wait for #iPad 2 also. The...,wait also sale
3,@sxsw I hope this year's festival isn't as cra...,hope years festival isnt crashy years iphone app
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,great stuff fri marissa mayer google tim oreil...


In [ ]:
#Performed text preprocessing including lowercasing, removal of URLs, mentions, special characters, and stopwords to clean the text data.

#Encoding

In [19]:
#use label encoder
from sklearn.preprocessing import LabelEncoder

In [20]:
le = LabelEncoder()

df['label'] = le.fit_transform(df['sentiment'])

In [21]:
print(df[['sentiment', 'label']].head())

          sentiment  label
0  Negative emotion      1
1  Positive emotion      3
2  Positive emotion      3
3  Negative emotion      1
4  Positive emotion      3


In [22]:
print(le.classes_)

["I can't tell" 'Negative emotion' 'No emotion toward brand or product'
 'Positive emotion']


In [ ]:
#Simplify Labels

In [23]:
df['sentiment'] = df['sentiment'].replace({
    "Positive emotion": "positive",
    "Negative emotion": "negative",
    "No emotion toward brand or product": "neutral",
    "I can't tell": "no_idea"
})

In [24]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment'])

In [25]:
print(le.classes_)

['negative' 'neutral' 'no_idea' 'positive']


#Tokenization & Padding

In [27]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [28]:
tokenizer = Tokenizer(num_words=5000)

In [29]:
tokenizer.fit_on_texts(df['clean_text'])

In [30]:
sequences = tokenizer.texts_to_sequences(df['clean_text'])

In [31]:
X = pad_sequences(sequences, maxlen=100)

In [32]:
y = df['label']

In [33]:
print(X.shape)
print(y.shape)

(9092, 100)
(9092,)


#Train-Test Split + Model Building (RNN / LSTM)

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_test.shape)

(7273, 100) (1819, 100)


In [35]:
#Build LSTM Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(input_dim=5000, output_dim=128, input_length=100),
    LSTM(64),
    Dense(4, activation='softmax')  # 4 classes
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [36]:
#Compile Model
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [37]:
#Train Model
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/5
114/114 ━━━━━━━━━━━━━━━━━━━━ 22s 147ms/step - accuracy: 0.6006 - loss: 0.9196 - val_accuracy: 0.6262 - val_loss: 0.8573
Epoch 2/5
114/114 ━━━━━━━━━━━━━━━━━━━━ 20s 147ms/step - accuracy: 0.6960 - loss: 0.7393 - val_accuracy: 0.6465 - val_loss: 0.8234
Epoch 3/5
114/114 ━━━━━━━━━━━━━━━━━━━━ 20s 140ms/step - accuracy: 0.7735 - loss: 0.5691 - val_accuracy: 0.6592 - val_loss: 0.8707
Epoch 4/5
114/114 ━━━━━━━━━━━━━━━━━━━━ 18s 157ms/step - accuracy: 0.8173 - loss: 0.4685 - val_accuracy: 0.6509 - val_loss: 0.9159
Epoch 5/5
114/114 ━━━━━━━━━━━━━━━━━━━━ 17s 145ms/step - accuracy: 0.8402 - loss: 0.4054 - val_accuracy: 0.6498 - val_loss: 1.0274


#Prediction and Evaluation

In [38]:
#Evaluate model
loss, acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", acc)

57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 0.6498 - loss: 1.0274
Test Accuracy: 0.6498075723648071


In [47]:
# Test multiple sentences
new_texts = [
    "I love Apple products",
    "I hate this Apple product",
    "This phone is okay",
    "I can't say anything about this brand"
]

# Preprocess
cleaned = [clean_text(t) for t in new_texts]

#Tokenize
seq = tokenizer.texts_to_sequences(cleaned)

# Pad
padded = pad_sequences(seq, maxlen=100)

# Predict
pred = model.predict(padded)

# Get class index
pred_classes = np.argmax(pred, axis=1)

# Convert to actual labels
pred_labels = le.inverse_transform(pred_classes)

#Display results
for text, label in zip(new_texts, pred_labels):
    print(f"Text: {text}")
    print(f"Predicted Sentiment: {label}")
    print("-" * 50)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 228ms/step
Text: I love Apple products
Predicted Sentiment: positive
--------------------------------------------------
Text: I hate this Apple product
Predicted Sentiment: negative
--------------------------------------------------
Text: This phone is okay
Predicted Sentiment: negative
--------------------------------------------------
Text: I can't say anything about this brand
Predicted Sentiment: negative
--------------------------------------------------


In [44]:
print(le.classes_)

['negative' 'neutral' 'no_idea' 'positive']


#BERT Model for Sentiment Classification

In [48]:
!pip install transformers torch

In [49]:
from transformers import pipeline

In [50]:
classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [51]:
texts = [
    "I love Apple products",
    "I hate this Apple product",
    "This phone is okay",
    "I can't say anything about this brand"
]

results = classifier(texts)

for text, res in zip(texts, results):
    print(f"Text: {text}")
    print(f"Predicted: {res['label']} (Confidence: {res['score']:.2f})")
    print("-" * 50)

Text: I love Apple products
Predicted: POSITIVE (Confidence: 1.00)
--------------------------------------------------
Text: I hate this Apple product
Predicted: NEGATIVE (Confidence: 1.00)
--------------------------------------------------
Text: This phone is okay
Predicted: POSITIVE (Confidence: 1.00)
--------------------------------------------------
Text: I can't say anything about this brand
Predicted: NEGATIVE (Confidence: 1.00)
--------------------------------------------------




*   BERT provides highly accurate predictions for positive and negative sentiments.
*   It performs better than LSTM in understanding context.



*   Limitation:It cannot directly classify into 4 classes (positive, negative, neutral, no idea).





